# Pawnee National Grassland Land Swap
## GBIF Grass Species Occurrence Mapping – Western Pawnee Grasslands
- **Objective:** For this notebook, the observations of 5 grasses, including blue grama(Bouteloua gracilis), buffalograss(Bouteloua dactyloides), sideoats grama(Bouteloua curtipendula), western wheatgrass(Pascopyrum smithii), and needle-and-thread (Hesperostipa comata), are pulled from [GBIF](https://www.gbif.org/) for each parcel within the Pawnee National Grassland boundary. A "ecological value" for these species will be generated for each parcel, and in the `parcel_matrix` notebook these ecological values will be appended to identify best land swaps. 

- **Objective goals:**
  - Download occurrence records for five target grass species using the GBIF API  
  - Convert tabular occurrence data into spatial point data (GeoDataFrames)  
  - Combine all species into a single dataset for analysis  
  - Clip occurrence points to the Western Pawnee Grasslands boundary  
  - Visualize species distributions using an interactive map  
  - Save processed spatial data and figures for downstream analysis

- **Author:** Kayleigh Ward
- **Author:**
- **Code review and/or edits:** Kayleigh Ward
- **Date:** April 9, 2026
- **Last date of revision:** April 22, 2026

---
### 🛠️ Prerequisites & Setup
**Mandatory Libraries:**
- `pandas` – tabular data handling  
- `geopandas` – spatial data processing and clipping  
- `pygbif` – GBIF API access for species occurrence downloads  
- `hvplot` / `holoviews` – interactive geospatial visualization  
- `shapely` – geometry creation (point conversion)  
- `os`, `pathlib` – file and directory management  

**Environment:**
- Conda environment (e.g., `geog`) with geospatial dependencies installed  
- Internet connection required for GBIF API downloads  
- Valid GBIF account credentials required for occurrence download requests  

**Data Sources:**
- GBIF occurrence data (downloaded dynamically via API)  
- Boundary shapefiles stored in: /data/boundaries/boundary-data-final/

**Related Notebooks:**
- Must run the `01_boundaries` notebook prior to this notebook  
- This notebook relies on the generated Western Pawnee boundary shapefile:
  - `pawnee_master_west.shp`

**Notes:**
- GBIF downloads are cached locally to avoid repeated API requests  
- All file paths are defined relative to the project root directory 

### 🏗️ Methodology
#### 1. Species Selection and Data Retrieval
Five grass species were selected and queried using the GBIF API via the `pygbif` library. For each species:
- A taxonomic backbone match was performed to obtain a GBIF taxon key  
- Occurrence records with coordinates were requested  
- Data were downloaded as compressed archives and extracted locally  

#### 2. Data Processing
- Occurrence records were loaded into pandas DataFrames  
- Relevant coordinate fields (`decimalLatitude`, `decimalLongitude`) were used to construct spatial points  
- Data were converted into GeoDataFrames using GeoPandas  
- All species GeoDataFrames were combined into a single dataset  

#### 3. Spatial Operations
- Boundary data were reprojected to a common coordinate reference system (EPSG:4326)  
- Occurrence points were clipped to the Western Pawnee boundary using spatial intersection  

#### 4. Visualization
- Interactive maps were created using `hvplot` with a basemap (Esri Imagery)  
- Species were symbolized by category and displayed with hover information  
- Map extent was constrained to the study area for clarity  

#### 5. Outputs
- Clipped GeoDataFrame of species occurrences  
- Interactive HTML map of species distributions  
- Saved outputs organized within project directories  

### Reproducibility Notes
- GBIF downloads are cached locally to avoid repeated API requests  
- File paths are structured relative to the project root directory  
- Boundary data must be generated prior to running this notebook  

### ⚡ Troubleshooting/Notes
* Make sure to login correctly to GBIF
* If login is incorrect you will get a 401 error when downloading data
* Install the pygbif package in terminal before importing libraries

# Libraries

In [5]:
# Path/file libraries
import os
import pathlib
import zipfile
import time
from glob import glob
from getpass import getpass

# Data handling 
import pandas as pd
import geopandas as gpd

# Web requests / data download
import requests

# Geospatial visualization 
import holoviews as hv
import hvplot.pandas
import cartopy.crs as ccrs

# GBIF API access
import pygbif.occurrences as occ
import pygbif.species as species
from getpass import getpass

# Primary Directory

In [ ]:
# Set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

Repo root: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project


# Secondary Directories from 01_boundaries notebook

In [ ]:
# Root dir
data_dir = os.path.join(repo_root, 'data')

# Main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

# Boundary dir written by 01_boundaries notebook
boundary_dir_final = os.path.join(boundary_dir, "boundary-data-final-west")

# MASTER
master_bound_west = os.path.join(boundary_dir_final, 'master_boundary')
master_bound_west_path = os.path.join(master_bound_west, 'pawnee_master_west.shp')
master_bound_west_gdf = gpd.read_file(master_bound_west_path)

# FEDERAL
federal_bound_west = os.path.join(boundary_dir_final, 'federal_boundary')
federal_bound_west_path = os.path.join(federal_bound_west, 'pawnee_fed_west.shp')
federal_bound_west_gdf = gpd.read_file(federal_bound_west_path)

# STATE
state_bound_west = os.path.join(boundary_dir_final, 'state_boundary')
state_bound_west_path = os.path.join(state_bound_west, 'pawnee_state_west.shp')
state_bound_west_gdf = gpd.read_file(state_bound_west_path)

# Output directory for clipped GBIF points
gbif_clipped_dir = os.path.join(data_dir, 'gbif_clipped')
os.makedirs(gbif_clipped_dir, exist_ok=True)

# Output directory for figures
gbif_grass_fig_dir = os.path.join(repo_root, 'figures', 'gbif')
os.makedirs(gbif_grass_fig_dir, exist_ok=True)

# GBIF Login for data download

In [ ]:
# Reset credentials
# If you happen to log in with the wrong credentials set this to True and rerun the cell
reset_credentials = False

# Make dictionary for GBIF username and pass
credentials = dict(
    GBIF_USER=(input, 'GBIF username:'),
    GBIF_PWD=(getpass, 'GBIF password'),
    GBIF_EMAIL=(input, 'GBIF email'),
)

# loop through credentials and enter them
for env_variable, (prompt_func, prompt_text) in credentials.items():

    if reset_credentials and (env_variable in os.environ):
        os.environ.pop(env_variable)

    if not env_variable in os.environ:
        os.environ[env_variable] = prompt_func(prompt_text)

## Download GBIF data for Pawnee National Grassland grasses

1. Create a helper function to pull data from GBIF
2. Define the species to download
3. Download data for the following grass species:
- blue grama(Bouteloua gracilis)
- buffalograss(Bouteloua dactyloides)
- sideoats grama(Bouteloua curtipendula)
- western wheatgrass(Pascopyrum smithii)
- needle-and-thread (Hesperostipa comata)

In [ ]:
# Function to download GBIF data for five grasses
def download_gbif_grass_species(species_name, folder_name, data_dir):
    """
    Download one GBIF species file, extract it, and read it into a DataFrame.
    Returns:
        df, csv_path, species_key
    """
    # Match species in GBIF backbone
    species_info = species.name_backbone(scientificName=species_name, kingdom="Plantae")

    # In the pygbif response, the key is nested under "usage"
    species_key = species_info.get("usage", {}).get("key")

    if species_key is None:
        raise ValueError(f"No usable GBIF key found for {species_name}. Returned: {species_info}")

    print(f"{species_name}: {species_key}")

    # Set species directory
    gbif_grasses_dir = os.path.join(data_dir, folder_name)
    os.makedirs(gbif_grasses_dir, exist_ok=True)

    # --- SKIP DOWNLOAD IF FILE EXISTS ---
    existing_files = [
        f for f in os.listdir(gbif_grasses_dir)
        if f.endswith(".csv")
    ]
    
    if existing_files:
        existing_path = os.path.join(gbif_grasses_dir, existing_files[0])
        print(f"{species_name}: using existing file")

        gbif_grasses_df = pd.read_csv(existing_path, sep="\t", low_memory=False)
        return gbif_grasses_df, existing_path, None

    # look for an extracted occurrence csv first
    existing_csvs = glob(os.path.join(gbif_grasses_dir, "*.csv"))
    if existing_csvs:
        gbif_grasses_path = existing_csvs[0]
        gbif_grasses_df = pd.read_csv(gbif_grasses_path, sep="\t", low_memory=False)
        return gbif_grasses_df, gbif_grasses_path, species_key

    # Submit query
    gbif_query = occ.download([
        f"speciesKey = {species_key}",
        "hasCoordinate = True",
    ])
    download_key = gbif_query[0]

    # Wait for the download to build
    status = occ.download_meta(download_key)["status"]
    while status != "SUCCEEDED":
        if status in {"CANCELLED", "KILLED", "FAILED"}:
            raise RuntimeError(f"GBIF download failed for {species_name}: {status}")
        time.sleep(5)
        status = occ.download_meta(download_key)["status"]

    # Download zipped Darwin Core archive
    occ.download_get(download_key, path=gbif_grasses_dir)

    # Find and unzip the download
    zip_matches = glob(os.path.join(gbif_grasses_dir, "*.zip"))
    if not zip_matches:
        raise FileNotFoundError(f"No GBIF zip download found for {species_name}")

    zip_path = zip_matches[0]
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(gbif_grasses_dir)

    # locate occurrence table
    csv_candidates = glob(os.path.join(gbif_grasses_dir, "*.csv")) + glob(os.path.join(gbif_grasses_dir, "*.txt"))
    if not csv_candidates:
        raise FileNotFoundError(f"No extracted occurrence table found for {species_name}")

    gbif_grasses_path = csv_candidates[0]
    gbif_grasses_df = pd.read_csv(gbif_grasses_path, sep="\t", low_memory=False)

    return gbif_grasses_df, gbif_grasses_path, species_key

In [ ]:
# Define the species to download
species_to_download = [
    {
        "species_name": "Bouteloua gracilis",
        "folder_name": "gbif_blue_grama"
    },
    {
        "species_name": "Bouteloua dactyloides",
        "folder_name": "gbif_buffalograss"
    },
    {
        "species_name": "Bouteloua curtipendula",
        "folder_name": "gbif_sideoats_grama"
    },
       {
        "species_name": "Pascopyrum smithii",
        "folder_name": "gbif_western_wheatgrass"
    },
       {
        "species_name": "Hesperostipa comata",
        "folder_name": "gbif_needle_and_thread"
    }
]

# Create an empty dictionary
gbif_grasses_data = {}

# loop through the download_gbif_species download
for item in species_to_download:
    gbif_grasses_df, gbif_grasses_path, species_key = download_gbif_grass_species(
        species_name=item["species_name"],
        folder_name=item["folder_name"],
        data_dir=data_dir
    )

    gbif_grasses_data[item["species_name"]] = {
        "df": gbif_grasses_df,
        "path": gbif_grasses_path,
        "species_key": species_key
    }

    print(f"\nLoaded {item['species_name']}")
    print(f"Path: {gbif_grasses_path}")
    print(gbif_grasses_df.head())

Bouteloua gracilis: 7861320


INFO:Your download key is 0014509-260423192947929
INFO:Download file size: 1065661 bytes
INFO:On disk at C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_blue_grama/0014509-260423192947929.zip



Loaded Bouteloua gracilis
Path: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_blue_grama\0014509-260423192947929.csv
      gbifID                            datasetKey  \
0  997428582  95c938a8-f762-11e1-a439-00145eb45e9a   
1  911520343  95c938a8-f762-11e1-a439-00145eb45e9a   
2  911490894  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
3  910457050  837acfc2-f762-11e1-a439-00145eb45e9a   
4  894993094  5678b0b3-450b-4513-82e1-2b32c3c50b54   

                                     occurrenceID  kingdom        phylum  \
0            972b4c5a-44bb-4a80-aad5-9b0ec7361780  Plantae  Tracheophyta   
1            9d5d292b-25af-4101-9edf-38cb78ecd3df  Plantae  Tracheophyta   
2  http://www.inaturalist.org/observations/299748  Plantae  Tracheophyta   
3                                   HSS:HSS:43949  Plantae  Tracheophyta   
4                                           17388  Plantae  Tracheophyta   

        class   order   family      genus             species  ...  \
0

INFO:Your download key is 0014535-260423192947929
INFO:Download file size: 495717 bytes
INFO:On disk at C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_buffalograss/0014535-260423192947929.zip



Loaded Bouteloua dactyloides
Path: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_buffalograss\0014535-260423192947929.csv
      gbifID                            datasetKey  \
0  930742158  0096dfc0-9925-47ef-9700-9b77814295f1   
1   91364638  85802736-f762-11e1-a439-00145eb45e9a   
2   91364625  85802736-f762-11e1-a439-00145eb45e9a   
3   91364619  85802736-f762-11e1-a439-00145eb45e9a   
4   91364608  85802736-f762-11e1-a439-00145eb45e9a   

                                        occurrenceID  kingdom        phylum  \
0  http://bioimages.vanderbilt.edu/ind-kaufmannm/...  Plantae  Tracheophyta   
1                                                NaN  Plantae  Tracheophyta   
2                                                NaN  Plantae  Tracheophyta   
3                                                NaN  Plantae  Tracheophyta   
4                                                NaN  Plantae  Tracheophyta   

        class   order   family      genus       

...


Loaded Bouteloua curtipendula
Path: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_sideoats_grama\0014558-260423192947929.csv
      gbifID                            datasetKey  \
0  930741485  0096dfc0-9925-47ef-9700-9b77814295f1   
1   91364529  85802736-f762-11e1-a439-00145eb45e9a   
2  911784935  de45bd59-54e6-4f62-9a15-b481af99fc48   
3  911784330  de45bd59-54e6-4f62-9a15-b481af99fc48   
4  899962454  50c9509d-22c7-4a22-a47d-8c48425ef4a7   

                                        occurrenceID  kingdom        phylum  \
0  http://bioimages.vanderbilt.edu/ind-baskauf/37...  Plantae  Tracheophyta   
1                                                NaN  Plantae  Tracheophyta   
2  IAVH:BICUB:COLOMBIA:PLANTAE:ESPECIMENPRESERVAD...  Plantae  Tracheophyta   
3  IAVH:BICUB:COLOMBIA:PLANTAE:ESPECIMENPRESERVAD...  Plantae  Tracheophyta   
4     http://www.inaturalist.org/observations/609435  Plantae  Tracheophyta   

        class   order   family      genus    

INFO:Your download key is 0014575-260423192947929
INFO:Download file size: 516 bytes
INFO:On disk at C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_western_wheatgrass/0014575-260423192947929.zip



Loaded Pascopyrum smithii
Path: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_western_wheatgrass\0014575-260423192947929.csv
Empty DataFrame
Columns: [gbifID, datasetKey, occurrenceID, kingdom, phylum, class, order, family, genus, species, infraspecificEpithet, taxonRank, scientificName, verbatimScientificName, verbatimScientificNameAuthorship, countryCode, locality, stateProvince, occurrenceStatus, individualCount, publishingOrgKey, decimalLatitude, decimalLongitude, coordinateUncertaintyInMeters, coordinatePrecision, elevation, elevationAccuracy, depth, depthAccuracy, eventDate, day, month, year, taxonKey, speciesKey, basisOfRecord, institutionCode, collectionCode, catalogNumber, recordNumber, identifiedBy, dateIdentified, license, rightsHolder, recordedBy, typeStatus, establishmentMeans, lastInterpreted, mediaType, issue]
Index: []

[0 rows x 50 columns]
Hesperostipa comata: 2702491


INFO:Your download key is 0014585-260423192947929
INFO:Download file size: 555834 bytes
INFO:On disk at C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_needle_and_thread/0014585-260423192947929.zip



Loaded Hesperostipa comata
Path: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_needle_and_thread\0014585-260423192947929.csv
      gbifID                            datasetKey  \
0  997427655  95c938a8-f762-11e1-a439-00145eb45e9a   
1  930741618  0096dfc0-9925-47ef-9700-9b77814295f1   
2  911520376  95c938a8-f762-11e1-a439-00145eb45e9a   
3  911520334  95c938a8-f762-11e1-a439-00145eb45e9a   
4  894993935  5678b0b3-450b-4513-82e1-2b32c3c50b54   

                                        occurrenceID  kingdom        phylum  \
0               4f58c867-7c59-44c4-ada3-22cb128711ad  Plantae  Tracheophyta   
1  http://bioimages.vanderbilt.edu/ind-baskauf/43...  Plantae  Tracheophyta   
2               88e5ec91-36ab-4092-9ffe-117f7ba1e337  Plantae  Tracheophyta   
3               ea6e1472-7bbc-4bb6-b2e5-8b28a50a499d  Plantae  Tracheophyta   
4                                               2572  Plantae  Tracheophyta   

        class   order   family         genus 

# Save the dataframe for each grass species
- gbif_blue_grama_df
- gbif_buffalograss_df
- gbif_sideoats_grama_df
- gbif_western_wheatgrass_df
- gbif_needle_and_thread_df

In [ ]:
# Save the five grass species as their own dataframes

gbif_blue_grama_df = gbif_grasses_data["Bouteloua gracilis"]["df"]

gbif_buffalograss_df = gbif_grasses_data["Bouteloua dactyloides"]["df"]

gbif_sideoats_grama_df = gbif_grasses_data["Bouteloua curtipendula"]["df"]

gbif_western_wheatgrass_df = gbif_grasses_data["Pascopyrum smithii"]["df"]

gbif_needle_and_thread_df = gbif_grasses_data["Hesperostipa comata"]["df"]

# Convert to geodataframes for mapping

In [12]:
# Make these spatial data frames (geodataframes)
for species_name, species_data in gbif_grasses_data.items():
    gbif_grasses_df = species_data["df"].dropna(
        subset=["decimalLongitude", "decimalLatitude"]
    ).copy()

    gbif_grasses_gdf = gpd.GeoDataFrame(
        gbif_grasses_df,
        geometry=gpd.points_from_xy(
            gbif_grasses_df["decimalLongitude"],
            gbif_grasses_df["decimalLatitude"]
        ),
        crs="EPSG:4326"
    )

    gbif_grasses_data[species_name]["gdf"] = gbif_grasses_gdf

    print(f"\nCreated GeoDataFrame for {species_name}")
    print(gbif_grasses_gdf.head())


Created GeoDataFrame for Bouteloua gracilis
      gbifID                            datasetKey  \
0  997428582  95c938a8-f762-11e1-a439-00145eb45e9a   
1  911520343  95c938a8-f762-11e1-a439-00145eb45e9a   
2  911490894  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
3  910457050  837acfc2-f762-11e1-a439-00145eb45e9a   
4  894993094  5678b0b3-450b-4513-82e1-2b32c3c50b54   

                                     occurrenceID  kingdom        phylum  \
0            972b4c5a-44bb-4a80-aad5-9b0ec7361780  Plantae  Tracheophyta   
1            9d5d292b-25af-4101-9edf-38cb78ecd3df  Plantae  Tracheophyta   
2  http://www.inaturalist.org/observations/299748  Plantae  Tracheophyta   
3                                   HSS:HSS:43949  Plantae  Tracheophyta   
4                                           17388  Plantae  Tracheophyta   

        class   order   family      genus             species  ...  \
0  Liliopsida  Poales  Poaceae  Bouteloua  Bouteloua gracilis  ...   
1  Liliopsida  Poales  Poaceae  B

# Save each grass species geodataframe

In [ ]:
# Save the grasses geodataframes

gbif_blue_grama_gdf = gbif_grasses_data["Bouteloua gracilis"]["gdf"]

gbif_buffalograss_gdf = gbif_grasses_data["Bouteloua dactyloides"]["gdf"]

gbif_sideoats_grama_gdf = gbif_grasses_data["Bouteloua curtipendula"]["gdf"]

gbif_western_wheatgrass_gdf = gbif_grasses_data["Pascopyrum smithii"]["gdf"]

gbif_needle_and_thread_gdf = gbif_grasses_data["Hesperostipa comata"]["gdf"]

# Composite them into one geodataframe

In [ ]:
# Combine into one gdf
gbif_grasses_gdf = gpd.GeoDataFrame(
    pd.concat(
        [
            gbif_blue_grama_gdf,
            gbif_buffalograss_gdf,
            gbif_sideoats_grama_gdf,
            gbif_western_wheatgrass_gdf,
            gbif_needle_and_thread_gdf
        ],
        ignore_index=True
    ),
    geometry="geometry",
    crs="EPSG:4326"
)

# Plot the composite geodataframe

In [ ]:
# Make a preliminary plot of the data
gbif_grasses_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Five grass species occurrences (GBIF)",
    frame_width=700,
    size=35
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (species)

In [ ]:
# Make a preliminary plot of the data with the Western Pawnee boundary
grass_points = gbif_grasses_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Five grass species occurrences (GBIF) at Western Pawnee",
    frame_width=700,
    size=35
)

grass_boundary = master_bound_west_gdf.to_crs("EPSG:4326").hvplot(
    geo=True,
    color=None,
    line_color="black",
    line_width=2
)

grass_points * grass_boundary

###---NOTE---###
# These maps may look the same but for this map you can zoom to the Pawnee West Boundary
# and you will see an outline has been added

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Points.I   :Points   [Longitude,Latitude]   (species)
   .Polygons.I :Polygons   [Longitude,Latitude]

In [ ]:
# Clip in a shared CRS
master_bound_west_wgs84 = master_bound_west_gdf.to_crs("EPSG:4326")
gbif_grasses_clipped = gpd.clip(gbif_grasses_gdf, master_bound_west_wgs84)

print(f"Original records: {len(gbif_grasses_gdf)}")
print(f"Clipped records: {len(gbif_grasses_clipped)}")
gbif_grasses_clipped.head()

Original records: 41947
Clipped records: 109


,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue,geometry
11149,1702479628,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/8718151,Plantae,Tracheophyta,Liliopsida,Poales,Poaceae,Bouteloua,Bouteloua gracilis,...,2017-11-07T05:31:46,CC_BY_NC_4_0,Kevin,Kevin,NaN,NaN,2026-04-18T10:05:45.374Z,StillImage,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...,POINT (-104.48809 40.62423)
41000,2242384191,89c53edb-0fac-4118-bdc0-d70ca50953dc,8bd62adc-ca67-40a5-bc1e-2ffb85deae82,Plantae,Tracheophyta,Liliopsida,Poales,Poaceae,Hesperostipa,Hesperostipa comata,...,NaN,CC0_1_0,Public Domain,Bruno Klinger; H. Zeiner,NaN,NaN,2026-04-06T17:23:38.733Z,StillImage,GEODETIC_DATUM_ASSUMED_WGS84;CONTINENT_DERIVED...,POINT (-104.53335 40.63143)
37460,4926332317,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/16825...,Plantae,Tracheophyta,Liliopsida,Poales,Poaceae,Hesperostipa,Hesperostipa comata,...,2023-06-19T06:36:18,CC_BY_NC_4_0,Matt Webb,Matt Webb,NaN,NaN,2026-04-18T09:03:24.185Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,POINT (-104.52347 40.63748)
37467,4926081681,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/16825...,Plantae,Tracheophyta,Liliopsida,Poales,Poaceae,Hesperostipa,Hesperostipa comata,...,2023-06-19T06:35:12,CC_BY_NC_4_0,Matt Webb,Matt Webb,NaN,NaN,2026-04-18T09:03:26.250Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,POINT (-104.52347 40.63748)
824,5840132071,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/30506...,Plantae,Tracheophyta,Liliopsida,Poales,Poaceae,Bouteloua,Bouteloua gracilis,...,2025-08-09T21:07:34,CC_BY_NC_4_0,mwillson,mwillson,NaN,NaN,2026-04-18T09:49:50.627Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,POINT (-104.34212 40.64299)


In [ ]:
# Save the clipped shapefile
gbif_clipped_path = os.path.join(gbif_clipped_dir, "gbif_grasses_clipped.shp")
gbif_grasses_clipped.to_file(gbif_clipped_path)

print(f"Saved clipped GBIF shapefile to: {gbif_clipped_path}")

C:\Users\kayle\AppData\Local\Temp\ipykernel_16792\4206173610.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gbif_grasses_clipped.to_file(gbif_clipped_path)
c:\Users\kayle\miniconda3\envs\geog\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'occurrenceID' to 'occurrence'
  ogr_write(
c:\Users\kayle\miniconda3\envs\geog\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'infraspecificEpithet' to 'infraspeci'
  ogr_write(
c:\Users\kayle\miniconda3\envs\geog\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'scientificName' to 'scientific'
  ogr_write(
c:\Users\kayle\miniconda3\envs\geog\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'verbatimScientificName' to 'verbatimSc'
  ogr_write(
c:\Users\kayle\miniconda3\envs\geog\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Norma

Saved clipped GBIF shapefile to: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\data\gbif_clipped\gbif_grasses_clipped.shp


# Plot the clipped grasses and save the figure

In [ ]:
# Plot clipped grass observations with the boundary
clipped_grass_points = gbif_grasses_clipped.hvplot(
    geo=True,
    tiles='EsriImagery',
    c='species',
    hover_cols=['species'],
    title='Clipped GBIF grass occurrences within Western Pawnee',
    frame_width=700,
    size=45
)

clipped_grass_boundary = master_bound_west_wgs84.hvplot(
    geo=True,
    color=None,
    line_color='yellow',
    line_width=2
)

clipped_grasses_plot = clipped_grass_points * clipped_grass_boundary
clipped_grasses_plot

# Save an interactive html version of the plot
clipped_grasses_plot_path = os.path.join(gbif_grass_fig_dir, 'gbif_grasses_clipped_map.html')
hv.save(clipped_grasses_plot, clipped_grasses_plot_path)

print(f'Saved clipped GBIF map to: {clipped_grasses_plot_path}')

Saved clipped GBIF map to: C:\Users\kayle\Desktop\earth-analytics\Pawnee-Grasslands-Project\figures\gbif\gbif_grasses_clipped_map.html
